In [8]:
from pipelines.readers.b3_indices_segmentos_setoriais import ReaderSQLB3IndicesSegmentosSetoriais

from yfinance import download
import plotly.express as px
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [2]:
reader_b3_segmentos_setoriais = ReaderSQLB3IndicesSegmentosSetoriais()
df_b3_segmentos_setoriais = reader_b3_segmentos_setoriais.read(indice="IBEP")

print(df_b3_segmentos_setoriais.shape)
df_b3_segmentos_setoriais.tail(3)

(71, 7)


,Indice,Data Referencia,Código,Ação,Tipo,Qtde. Teórica,Part. (%)
68,IBEP,2026-07-06,VIVT3,TELEF BRASIL,ON EJ,707125712,1.166
69,IBEP,2026-07-06,WEGE3,WEG,ON NM,1485954732,3.278
70,IBEP,2026-07-06,YDUQ3,YDUQS PART,ON NM,261365845,0.112


In [3]:
drawdown = {}

for codigo in df_b3_segmentos_setoriais['Código']:
    
    serie_precos = download(codigo + ".SA", period="10y", interval="1d", progress=False, auto_adjust=False).droplevel(1, axis=1)
    
    preco_maximo = serie_precos['Adj Close'].max()
    
    preco_atual = serie_precos['Adj Close'].iloc[-1]
    
    drawdown_atual = (preco_atual - preco_maximo) / preco_maximo * 100
    
    drawdown[codigo] = drawdown_atual


In [4]:
df = df_b3_segmentos_setoriais.copy()

df["drawdown"] = df["Código"].map(drawdown)

print(df.shape)

df.tail(3)

(71, 8)


,Indice,Data Referencia,Código,Ação,Tipo,Qtde. Teórica,Part. (%),drawdown
68,IBEP,2026-07-06,VIVT3,TELEF BRASIL,ON EJ,707125712,1.166,-15.949626
69,IBEP,2026-07-06,WEGE3,WEG,ON NM,1485954732,3.278,-16.256361
70,IBEP,2026-07-06,YDUQ3,YDUQS PART,ON NM,261365845,0.112,-81.908784


In [5]:
plot_df = (
    df[["Código", "drawdown"]]
    .dropna()
    .sort_values("drawdown")
)

fig = px.bar(
    plot_df,
    x="Código",
    y="drawdown",
    color_discrete_sequence=["firebrick"],
    labels={
        "drawdown": "Drawdown (%)",
        "Código": "Empresa"
    },
    title="Drawdown Máximo por Empresa"
)

fig.add_hline(y=0, line_color="black")

fig.update_layout(
    height=500,
    width=1200,
    xaxis_tickangle=-90
)

fig.show()

In [ ]:
# Cálculo do drawdown para uma ação específica

serie_precos = download(ticker:="BBDC4.SA", period="10y", interval="1d", progress=False, auto_adjust=False).droplevel(1, axis=1)

preco_maximo = serie_precos["Adj Close"].expanding().max()
drawdown = (serie_precos["Adj Close"] - preco_maximo) / preco_maximo * 100
max_drawdown = drawdown.min()

In [ ]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.7, 0.3]
)

fig.add_trace(
    go.Scatter(
        x=serie_precos.index,
        y=serie_precos["Adj Close"],
        name="Preço"
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=drawdown.index,
        y=drawdown,
        name="Drawdown",
        fill="tozeroy",
        fillcolor="rgba(225,112,85,0.4)",
        line=dict(color="rgb(225,112,85)", width=1.5)
    ),
    row=2, col=1
)

fig.update_layout(
    height=600,
    width=1200
)

fig.show()